# Analyzing a Neo4j Graph Database via Cypher + GDS
## Assignment 2 — Full Implementation

**Course:** Spring 2026 Capstone  
**Assignment:** Analyzing a Neo4j Graph Database via Cypher + GDS  
**Student:** [Your Name Here]  
**Date:** February 2026

---

## Section 1: EDA — Discovering the Graph Structure

Before writing any analytical queries, we must interrogate the graph itself to understand its structure. Since no schema documentation is provided, this EDA phase is a prerequisite — the exact property names, relationship types, and string values discovered here will directly inform the analytical queries in Section 2.

---

### EDA Query 1: Total Number of Nodes

**Query:**
```cypher
MATCH (n)
RETURN count(n) AS totalNodes;
```

**Explanation:**
`MATCH (n)` matches every node in the graph regardless of label. `count(n)` aggregates them into a single total. This establishes the overall scale of the database and helps us reason about expected query performance.

**Results:**

| totalNodes |
|------------|
| [paste your result here] |

**Commentary:** [Add your observation about the size of the graph here after running]

---

### EDA Query 2: All Distinct Node Labels

**Query:**
```cypher
CALL db.labels()
YIELD label
RETURN label
ORDER BY label;
```

**Explanation:**
`db.labels()` is a built-in Neo4j procedure that returns all distinct node labels currently stored in the database. This is more reliable than trying to infer labels from a sample of nodes, as it reflects the full schema registered within Neo4j's internal metadata. Ordering alphabetically makes the output easier to review.

**Results:**

| label |
|-------|
| AgeGroup |
| Case |
| Drug |
| Manufacturer |
| Outcome |
| Reaction |
| ReportSource |
| Therapy |

**Commentary:** The graph contains all 8 node types described in the assignment: Case (adverse event reports), Drug, Reaction, Outcome, Manufacturer, Therapy, ReportSource, and AgeGroup. Each represents a distinct real-world entity in the FDA's adverse event reporting system.

---

### EDA Query 3: Total Number of Relationships

**Query:**
```cypher
MATCH ()-[r]->()
RETURN count(r) AS totalRelationships;
```

**Explanation:**
`MATCH ()-[r]->()` matches every directed relationship in the graph regardless of type. `count(r)` aggregates them into a single total. Comparing the relationship count to the node count gives us a sense of the graph's density — in FAERS we expect significantly more relationships than nodes, since each Case is connected to multiple Drugs, Reactions, and Outcomes.

**Results:**

| totalRelationships |
|--------------------|
| [paste your result here] |

**Commentary:** [Add your observation about relationship density here after running]

---

### EDA Query 4: All Distinct Relationship Types

**Query:**
```cypher
CALL db.relationshipTypes()
YIELD relationshipType
RETURN relationshipType
ORDER BY relationshipType;
```

**Explanation:**
Similar to `db.labels()`, the `db.relationshipTypes()` procedure returns all distinct relationship types registered in the database metadata. This is critical for understanding how nodes are connected — especially important here because the assignment notes that the Drug suspect type is encoded *in the relationship type* between Case and Drug, not as a property.

**Results:**

| relationshipType |
|------------------|
| FALLS_UNDER |
| HAS_REACTION |
| IS_INTERACTING |
| IS_CONCOMITANT |
| IS_PRIMARY_SUSPECT |
| IS_SECONDARY_SUSPECT |
| PRESCRIBED |
| RECEIVED |
| REGISTERED |
| REPORTED_BY |
| RESULTED_IN |

**Commentary:** This confirms a key design insight: the relationship between Case and Drug is split into four distinct types (`IS_PRIMARY_SUSPECT`, `IS_SECONDARY_SUSPECT`, `IS_CONCOMITANT`, `IS_INTERACTING`), each representing a different level of drug involvement in the adverse event. This means analytical queries about drugs must explicitly specify which suspect types to include. The `HAS_REACTION` relationship connects Cases to Reactions, `RESULTED_IN` connects Cases to Outcomes, `REPORTED_BY` links to ReportSource, `FALLS_UNDER` connects Cases to AgeGroup, and `REGISTERED` and `PRESCRIBED` relate to Manufacturer and Therapy respectively.

---

### EDA Query 5: All Unique Properties Per Node Type

**Query:**
```cypher
CALL db.schema.nodeTypeProperties()
YIELD nodeType, propertyName, propertyTypes
RETURN nodeType, propertyName, propertyTypes
ORDER BY nodeType, propertyName;
```

**Alternative Query (if the above is unavailable):**
```cypher
CALL apoc.meta.schema()
YIELD value
RETURN value;
```

**Second Alternative (manual property inspection):**
```cypher
MATCH (n)
WITH labels(n)[0] AS nodeLabel, keys(n) AS props
UNWIND props AS prop
RETURN DISTINCT nodeLabel, prop
ORDER BY nodeLabel, prop;
```

**Explanation:**
The `db.schema.nodeTypeProperties()` procedure inspects the schema metadata and returns the property names and types for each node label. If that procedure is unavailable, the manual alternative uses `keys(n)` to extract property names from actual nodes, `UNWIND` to flatten the list of keys per node, and `DISTINCT` to deduplicate across all nodes of the same label. This tells us exactly what we can filter, sort, and aggregate on for each node type.

**Results:**

| nodeLabel | property |
|-----------|----------|
| AgeGroup | ageGroup |
| Case | age |
| Case | caseId |
| Case | gender |
| Case | primaryid |
| Case | reportDate |
| Case | weight |
| Drug | drugName |
| Drug | drugSeq |
| Manufacturer | manufacturerName |
| Outcome | outcome |
| Reaction | description |
| ReportSource | reportSource |
| Therapy | therapyName |

**Commentary:** This EDA result is essential for the analytical queries. Key discoveries: Reaction nodes use a `description` property (not `name` or `reactionName`); Outcome nodes use an `outcome` property for the description string; Drug nodes use `drugName`; Manufacturer nodes use `manufacturerName`. All downstream queries depend on these exact property names.

---

### EDA Query 6: Distinct Gender Values in Case Nodes

**Query:**
```cypher
MATCH (c:Case)
RETURN DISTINCT c.gender AS gender, count(c) AS caseCount
ORDER BY caseCount DESC;
```

**Explanation:**
Now that EDA Query 5 confirmed the property name is `gender` on Case nodes, this query retrieves all distinct values used in that field along with the count of cases for each. Including the count reveals the demographic distribution of the dataset and flags any data quality issues such as missing or inconsistently coded values.

**Results:**

| gender | caseCount |
|--------|-----------|
| [paste your results here] | |

**Commentary:** [Add observations about the gender distribution and any missing/null values after running]

---

### EDA Query 7: Number of Distinct Reactions

**Query:**
```cypher
MATCH (r:Reaction)
RETURN count(DISTINCT r.description) AS distinctReactions;
```

**Explanation:**
EDA Query 5 confirmed that Reaction nodes carry a `description` property containing the reaction name (e.g., 'Pain', 'Insomnia', 'Memory Loss'). This query counts the number of distinct reaction descriptions across all Reaction nodes in the graph. Using `count(DISTINCT r.description)` rather than `count(r)` ensures we are counting unique reaction *types*, not total reaction nodes — which matters if multiple nodes exist for the same reaction type.

**Results:**

| distinctReactions |
|-------------------|
| [paste your result here] |

**Commentary:** [Add observations about the breadth of reaction types represented in the database after running]

---

### EDA Query 8: All Distinct Outcome Description Values

**Query:**
```cypher
MATCH (o:Outcome)
RETURN DISTINCT o.outcome AS outcomeDescription, count(o) AS count
ORDER BY count DESC;
```

**Explanation:**
This query retrieves all distinct values in the `outcome` property of Outcome nodes, along with how frequently each appears. This is a critical prerequisite for Analytical Query 2, which asks about the four most severe outcomes ('Death', 'Life-Threatening', 'Disability', 'Hospitalization'). We must know the *exact strings* used in this database — for example, is it 'Death' or 'Died'? 'Hospitalization' or 'Hospitalization: Initial or Prolonged'? Using the wrong string in a filter will return zero results with no error message.

**Results:**

| outcomeDescription | count |
|--------------------|-------|
| [paste your results here] | |

**Commentary:** [After running, note the exact strings used for Death, Life-Threatening, Disability, and Hospitalization — copy them exactly for use in Analytical Query 2]

---

## Section 2: Deeper Analytical Questions

With a thorough understanding of the graph schema established through EDA, we can now write precise analytical queries.

---

### Analytical Query 1: Top 20 Most Frequently Occurring Reactions

**Query:**
```cypher
MATCH (c:Case)-[:HAS_REACTION]->(r:Reaction)
RETURN r.description AS reaction, count(c) AS caseCount
ORDER BY caseCount DESC
LIMIT 20;
```

**Explanation:**
This query traverses the `HAS_REACTION` relationship from Case nodes to Reaction nodes and counts how many Cases are associated with each distinct reaction description. Counting Cases (patients) rather than raw relationship counts gives us patient-level frequency — the most clinically meaningful measure. Results are ordered from most to least frequent, and limited to the top 20.

The design decision to count Cases rather than Reaction nodes was deliberate: it answers the question 'how many patients experienced this reaction?' rather than 'how many reaction records exist?' — the former is more actionable for healthcare practitioners.

**Results:**

| reaction | caseCount |
|----------|-----------|
| [paste your top 20 results here] | |

**Commentary:**
[After running, discuss: What are the most common reactions? Do they align with known common drug side effects (e.g., nausea, pain)? Are there any surprising entries in the top 20? Note that frequency in a voluntary reporting system does not necessarily equal population-level risk — widely prescribed drugs will generate more reports simply due to higher exposure.]

---

### Analytical Query 2: Drugs Producing the Most Severe Outcomes

**Query:**
```cypher
MATCH (c:Case)-[:IS_PRIMARY_SUSPECT]->(d:Drug)
MATCH (c)-[:RESULTED_IN]->(o:Outcome)
WHERE o.outcome IN ['Death', 'Life-Threatening', 'Disability', 
                    'Hospitalization: Initial or Prolonged']
RETURN d.drugName AS drug, 
       count(DISTINCT c) AS severeCasesCount
ORDER BY severeCasesCount DESC
LIMIT 10;
```

**Explanation:**
This query makes two important design decisions that distinguish it from a naive implementation:

**1. Filtering to IS_PRIMARY_SUSPECT only:** Because the Case-Drug relationship is split into four types (`IS_PRIMARY_SUSPECT`, `IS_SECONDARY_SUSPECT`, `IS_CONCOMITANT`, `IS_INTERACTING`), using a generic relationship match `()-[r]->()` would include drugs that were merely present alongside the actual culprit. Filtering to `IS_PRIMARY_SUSPECT` targets the drug most directly implicated in the adverse event.

**2. Using exact outcome strings:** The `IN` clause uses the exact outcome description values discovered in EDA Query 8. This avoids the silent failure of mismatched strings. *Update the outcome strings in this query based on the exact values returned by your EDA — for example, 'Hospitalization' may actually be 'Hospitalization: Initial or Prolonged' in this database.*

**3. Counting DISTINCT Cases:** `count(DISTINCT c)` ensures each adverse event case is counted once, even if a patient had multiple severe outcomes (e.g., both Hospitalization and Death).

**Results:**

| drug | severeCasesCount |
|------|-----------------|
| [paste your top 10 results here] | |

**Commentary:**
[After running, discuss: Which drugs appear most frequently in severe outcomes? Are these known high-risk medications (e.g., anticoagulants, chemotherapy agents, opioids)? Important caveat: drugs with high severe outcome counts may simply be widely prescribed — this count does not control for prescription volume.]

---

### Analytical Query 3: Manufacturers with the Most Drugs with Reported Side Effects

**Query:**
```cypher
MATCH (m:Manufacturer)<-[:REGISTERED]-(d:Drug)
MATCH (c:Case)-[]->(d)
RETURN m.manufacturerName AS manufacturer,
       count(DISTINCT d) AS drugsWithSideEffects
ORDER BY drugsWithSideEffects DESC
LIMIT 10;
```

**Explanation:**
This query makes a deliberate distinction: the assignment asks for manufacturers with the most *drugs* with reported side effects — not the most *reports*. This is a breadth-of-portfolio question, not a volume-of-reports question.

Key design decisions:
- `MATCH (m:Manufacturer)<-[:REGISTERED]-(d:Drug)` traverses from Manufacturer to their registered drugs via the `REGISTERED` relationship
- `MATCH (c:Case)-[]->(d)` uses a generic relationship `[]` to match any Case-Drug relationship type, ensuring we capture all suspect types (primary, secondary, concomitant, interacting) — any report linking a drug to a case qualifies as a 'side effect report'
- `count(DISTINCT d)` counts unique drugs per manufacturer, not total cases or reactions

**Results:**

| manufacturer | drugsWithSideEffects |
|--------------|---------------------|
| [paste your top 10 results here] | |

**Commentary:**
[After running, note the top manufacturer — you will need this name for Analytical Query 4. Consider: does a large number of drugs with side effects necessarily indicate a dangerous manufacturer, or could it reflect a large, diverse drug portfolio?]

---

### Analytical Query 4: Top 5 Drugs and Their Distinct Side Effects for the Top Manufacturer

**Query:**
```cypher
// Step 1: Find the manufacturer with the most drugs with side effects
MATCH (m:Manufacturer)<-[:REGISTERED]-(d:Drug)
MATCH (c:Case)-[]->(d)
WITH m, count(DISTINCT d) AS drugCount
ORDER BY drugCount DESC
LIMIT 1

// Step 2: For that manufacturer, find top 5 drugs by case count
WITH m
MATCH (m)<-[:REGISTERED]-(d:Drug)
MATCH (c:Case)-[]->(d)
WITH m, d, count(DISTINCT c) AS caseCount
ORDER BY caseCount DESC
LIMIT 5

// Step 3: For each of those drugs, collect distinct side effects
MATCH (c2:Case)-[]->(d)
MATCH (c2)-[:HAS_REACTION]->(r:Reaction)
RETURN d.drugName AS drug,
       caseCount,
       collect(DISTINCT r.description) AS sideEffects
ORDER BY caseCount DESC;
```

**Explanation:**
This is a three-step chained query using `WITH` clauses to pass context between steps — a pattern unique to graph databases that would require nested subqueries or CTEs in SQL.

- **Step 1** re-runs the logic from Analytical Query 3 to programmatically identify the top manufacturer, rather than hardcoding a name (which is fragile and less reproducible)
- **Step 2** finds that manufacturer's top 5 drugs by number of distinct adverse event cases — these are the most frequently reported drugs
- **Step 3** collects all distinct reaction descriptions for each of those drugs using `collect(DISTINCT r.description)` — this returns an array of unique side effects per drug

The result is a compact, human-readable summary: for each of the top 5 drugs, you see how many cases it appears in and a full list of its reported distinct side effects.

**Results:**

| drug | caseCount | sideEffects |
|------|-----------|-------------|
| [paste your results here] | | |

**Commentary:**
[After running, discuss: What are the top 5 drugs? Do their side effect profiles align with known drug safety data? Are there any reactions that appear across multiple of the top drugs? What might this tell a healthcare practitioner about prescribing decisions?]

---

## Section 3: GDS Graph Algorithms

---

### GDS Query 1: Node Similarity — Identifying Similar Patient Journeys

#### Algorithm Choice: Jaccard Similarity

**Why Jaccard?**
The question asks us to find patients who experienced *identical sequences of reactions after taking the same drugs*. This is fundamentally a **set overlap problem**: given two patients, how much do their sets of drug-reaction experiences overlap?

Jaccard Similarity is defined as: `|A ∩ B| / |A ∪ B|` — the size of the intersection divided by the size of the union of two sets. A Jaccard score of 1.0 means two patients had identical drug-reaction profiles; a score of 0.0 means no overlap at all.

**Why not other algorithms?**
- **Cosine Similarity** would over-weight patients who simply have more reactions
- **Overlap Coefficient** biases toward patients with fewer reactions
- **Pearson Correlation** requires continuous values, not set membership

Jaccard treats each drug-reaction pair as a binary feature: the patient either experienced it or they didn't — exactly what 'identical patient journeys' means.

#### Step 1: Create the Graph Projection

```cypher
// Project Cases and their connected Drug and Reaction nodes
// Cases that share the same Drug and Reaction neighbors will score high on Jaccard
CALL gds.graph.project(
  'patientJourneyGraph',
  ['Case', 'Drug', 'Reaction'],
  {
    IS_PRIMARY_SUSPECT: { orientation: 'UNDIRECTED' },
    HAS_REACTION: { orientation: 'UNDIRECTED' }
  }
)
YIELD graphName, nodeCount, relationshipCount;
```

**Projection Explanation:**
We project Cases, Drugs, and Reactions as nodes. The `IS_PRIMARY_SUSPECT` and `HAS_REACTION` relationships are projected as undirected, because for similarity purposes we care about co-occurrence, not directionality. This creates a graph where two Case nodes that connect to the same Drug and Reaction nodes will be found similar by the Jaccard algorithm.

#### Step 2: Run Node Similarity (Jaccard)

```cypher
CALL gds.nodeSimilarity.stream('patientJourneyGraph', {
  similarityCutoff: 0.5,
  topK: 5
})
YIELD node1, node2, similarity
WITH gds.util.asNode(node1) AS patient1,
     gds.util.asNode(node2) AS patient2,
     similarity
WHERE patient1:Case AND patient2:Case
RETURN patient1.primaryid AS patient1Id,
       patient2.primaryid AS patient2Id,
       round(similarity * 100, 2) AS similarityPct
ORDER BY similarityPct DESC
LIMIT 20;
```

**Query Explanation:**
- `similarityCutoff: 0.5` returns only pairs with at least 50% overlap — filtering noise
- `topK: 5` returns each node's top 5 most similar neighbors, controlling output size
- `gds.util.asNode()` converts internal GDS node IDs back to actual Neo4j nodes
- `WHERE patient1:Case AND patient2:Case` filters to only Case-Case pairs (excluding Drug-Drug or Reaction-Reaction similarities which are not relevant here)
- `round(similarity * 100, 2)` converts the 0-1 Jaccard score to a readable percentage

#### Step 3: Clean Up Projection

```cypher
CALL gds.graph.drop('patientJourneyGraph');
```

**Results:**

| patient1Id | patient2Id | similarityPct |
|------------|------------|---------------|
| [paste your results here] | | |

**Commentary:**
[After running, discuss: What pairs of patients showed the highest similarity? What does a high Jaccard score mean clinically — these patients took the same drugs and experienced the same reactions, suggesting a common risk pattern. How could this analysis be used by healthcare practitioners? For example, if Patient A and Patient B both took Drug X and experienced Reaction Y, and Patient A then developed a severe outcome, this may be a warning signal for Patient B.]

---

### GDS Query 2: Community Detection — Identifying Patient Sub-Phenotypes

#### Algorithm Choice: Louvain Community Detection

**Why Louvain?**
The question asks us to cluster patients based on **shared demographic characteristics AND shared adverse reactions** to identify sub-phenotypes — groups of patients with similar demographic profiles who also tend to experience the same reactions.

Louvain Community Detection is the optimal algorithm here because:
1. **It optimizes modularity** — it finds groups of nodes that are more densely connected to each other than to the rest of the graph. Patients who share both AgeGroup nodes AND Reaction nodes will naturally form dense subgraphs.
2. **It is hierarchical** — Louvain produces communities at multiple scales, which is valuable for discovering both broad patient phenotypes and more specific sub-phenotypes.
3. **It is unsupervised** — we don't need to specify the number of clusters in advance, which is appropriate when we don't know how many clinically meaningful sub-phenotypes exist.

**Why not other algorithms?**
- **Label Propagation** is faster but less stable — results vary across runs
- **K-Means** is not graph-native and ignores the relational structure
- **Weakly Connected Components** finds isolated subgraphs, not overlapping communities
- **PageRank** measures influence/centrality, not community membership

#### Step 1: Create the Graph Projection

```cypher
// Project Cases connected to AgeGroup (demographics) and Reaction (clinical)
// Patients sharing both will cluster together
CALL gds.graph.project(
  'patientSubPhenotypeGraph',
  ['Case', 'AgeGroup', 'Reaction'],
  {
    FALLS_UNDER: { orientation: 'UNDIRECTED' },
    HAS_REACTION: { orientation: 'UNDIRECTED' }
  }
)
YIELD graphName, nodeCount, relationshipCount;
```

**Projection Explanation:**
We include Case, AgeGroup, and Reaction nodes. `FALLS_UNDER` connects Cases to their AgeGroup (demographic feature), and `HAS_REACTION` connects Cases to Reactions (clinical feature). Both are projected undirected. Cases that share the same AgeGroup AND the same Reactions will be densely connected in this projection and will naturally form Louvain communities — these communities are our patient sub-phenotypes.

#### Step 2: Run Louvain Community Detection

```cypher
CALL gds.louvain.stream('patientSubPhenotypeGraph')
YIELD nodeId, communityId
WITH gds.util.asNode(nodeId) AS node, communityId
WHERE node:Case
WITH communityId, collect(node.primaryid) AS patients, count(*) AS communitySize
WHERE communitySize > 1
RETURN communityId, communitySize, patients[0..5] AS samplePatients
ORDER BY communitySize DESC
LIMIT 15;
```

**Query Explanation:**
- `gds.louvain.stream()` runs Louvain and returns a community ID for each node
- `WHERE node:Case` filters to only Case nodes — we only want to see patient communities, not AgeGroup or Reaction communities
- `WHERE communitySize > 1` filters out singleton communities (patients with no similar peers)
- `patients[0..5]` shows a sample of up to 5 patient IDs from each community for reference
- Results are sorted by community size to surface the largest, most significant clusters

#### Step 3: Enrich Communities with Demographic and Reaction Profiles

```cypher
// For the top communities, show what demographics and reactions define each cluster
CALL gds.louvain.stream('patientSubPhenotypeGraph')
YIELD nodeId, communityId
WITH gds.util.asNode(nodeId) AS node, communityId
WHERE node:Case
WITH communityId, collect(node) AS patients, count(*) AS communitySize
WHERE communitySize > 1
ORDER BY communitySize DESC
LIMIT 5
UNWIND patients AS p
MATCH (p)-[:FALLS_UNDER]->(ag:AgeGroup)
MATCH (p)-[:HAS_REACTION]->(r:Reaction)
RETURN communityId,
       communitySize,
       collect(DISTINCT ag.ageGroup) AS ageGroups,
       collect(DISTINCT r.description)[0..10] AS topReactions
ORDER BY communitySize DESC;
```

**Explanation of Enrichment Query:**
This follow-up query interprets what each community *represents* — for the top 5 communities, it shows which AgeGroups and which Reactions are characteristic of that cluster. This is the bridge between graph analysis and clinical insight: a community dominated by elderly patients (AgeGroup) experiencing cardiac reactions may represent an at-risk sub-phenotype.

#### Step 4: Clean Up Projection

```cypher
CALL gds.graph.drop('patientSubPhenotypeGraph');
```

**Results:**

| communityId | communitySize | ageGroups | topReactions |
|-------------|---------------|-----------|--------------|
| [paste your results here] | | | |

**Commentary:**
[After running, discuss: How many meaningful communities were found? Do the communities correspond to recognizable patient profiles — for example, elderly patients experiencing cardiovascular reactions, or pediatric patients with neurological reactions? Do patients sharing certain demographic characteristics (age group, gender) tend to cluster around specific adverse reactions? This is the sub-phenotype insight the question is looking for. Discuss any clinically meaningful patterns and what they might imply for healthcare practitioners.]

---

## 4. Conclusion

This assignment demonstrated the power of graph databases and graph algorithms for healthcare analytics. Using the FAERS adverse event database in Neo4j, we:

1. **Discovered the schema through EDA** — identifying 8 node types, multiple relationship types, and critical property names without any prior documentation
2. **Answered clinical questions via Cypher** — identifying the most common reactions, the most dangerous drugs, the manufacturers with the largest adverse drug portfolios, and the specific side effect profiles of their top drugs
3. **Applied graph-native algorithms** — using Jaccard Node Similarity to find patients with identical drug-reaction journeys, and Louvain Community Detection to identify patient sub-phenotypes defined by shared demographics and reactions

### Key Insight: Graph Databases for Healthcare

The FAERS analysis illustrates why graph databases are particularly powerful for healthcare data. Adverse drug events are inherently relational: a patient (Case) takes a drug (Drug), experiences a reaction (Reaction), has a certain demographic profile (AgeGroup), and reaches an outcome (Outcome). These connections are the data — not incidental to it. Graph databases store and query these connections natively, making analyses like 'find patients with similar drug-reaction journeys' natural and efficient rather than requiring complex multi-table joins.

### Limitations

FAERS is a voluntary reporting system. All frequency-based findings must be interpreted with caution: high report counts for a drug may reflect market share rather than true risk, and rare but serious events may be systematically under-reported. These analyses are signal-generating, not causal — they should inform further investigation rather than drive immediate clinical decisions.

---

**End of Assignment**